Take Home Exercise #2

1.What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

In [0]:
df_drivers = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/drivers.csv",
    header=True,
    inferSchema=True
)

In [0]:
df_pit = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/pit_stops.csv",
    header=True,
    inferSchema=True
)

In [0]:
display(df_pit)

To answer this question, I will use the f1 pit_stops dataset and group the data by raceId and driverId. Then, I will calculate the average, minimum, and maximum pit stop time in milliseconds for each driver in each race.

After that, I joined the result with the drivers dataset to add each driver’s name, which makes the output easier to read and interpret.

In [0]:
q1_df = df_pit.groupBy("raceId", "driverId") \
    .agg(
        avg("milliseconds").alias("avg_pit"),
        min("milliseconds").alias("fastest_pit"),
        max("milliseconds").alias("slowest_pit")
    )

In [0]:
q1_df = q1_df.join(
    df_drivers.select("driverId", "forename", "surname"),
    on="driverId",
    how="left"
)


In [0]:
from pyspark.sql.functions import concat_ws

In [0]:
q1_df = q1_df.withColumn(
    "driver_name",
    concat_ws(" ", "forename", "surname")
).select(
    "raceId",
    "driverId",
    "driver_name",
    "avg_pit",
    "fastest_pit",
    "slowest_pit"
).orderBy("raceId", "driverId")

In [0]:
display(q1_df)

2. Rank order by finishing position the average time spent at the pit stop in each race.

To answer this question, I will combine the pit_stops dataset with the results dataset using raceId and driverId. This allows me to match each driver’s pit stop times with their finishing position in each race. Then, I will group the data by raceId and finishing position and calculate the average pit stop time in milliseconds for each finishing position within each race. Finally, I will sort the results by raceId and position to show the ranking order clearly.

In [0]:
from pyspark.sql.functions import col, when, avg

In [0]:
df_results_q2 = df_results.select("raceId", "driverId", "position") \
    .withColumn(
        "position",
        when(col("position") == "\\N", None).otherwise(col("position"))
    ) \
    .withColumn("position", col("position").cast("int")) \
    .filter(col("position").isNotNull())

In [0]:
q2_df = df_pit.join(
    df_results_q2,
    on=["raceId", "driverId"],
    how="inner"
)

q2_df = q2_df.groupBy("raceId", "position") \
    .agg(
        avg("milliseconds").alias("avg_pit_by_position")
    ) \
    .orderBy("raceId", "position")

display(q2_df)

3.Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

To fill in the missing driver codes, I use the drivers dataset and create a rule based on the driver's surname. If the code is missing, I replace it with the first three letters of the surname in uppercase. For example, Alonso becomes ALO. I chose this method because it is simple, consistent, and matches the example given in the question. Existing non-missing codes are kept unchanged.

In [0]:
from pyspark.sql.functions import col, when, upper, substring

df_drivers = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/drivers.csv",
    header=True,
    inferSchema=True
)

q3_df = df_drivers.withColumn(
    "code_filled",
    when(
        (col("code").isNull()) | (col("code") == "\\N"),
        upper(substring(col("surname"), 1, 3))
    ).otherwise(col("code"))
)

display(
    q3_df.select(
        "driverId",
        "forename",
        "surname",
        "code",
        "code_filled"
    ).orderBy("driverId")
)

I used withColumn() to create a new column called code_filled. Then, I used when() to check whether the original code was missing, either as a null value or as "\N". If the code was missing, I generated a replacement by taking the first three letters of the surname with substring() and converting them to uppercase with upper(). If the code was already present, I kept the original value using otherwise(). This approach preserves existing information while filling in missing values in a consistent way.

4.Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

To answer this question, I define age as the driver’s age in completed years on the date of the race. I use the drivers dataset for each driver’s date of birth (dob) and the races dataset for the race date. Then, I join these datasets with the results dataset so that each driver can be matched to each race they participated in. After creating the Age column, I identify the youngest and oldest driver in each race by comparing ages within the same race.

In [0]:
from pyspark.sql.functions import col, floor, datediff, row_number
from pyspark.sql.window import Window

# read datasets
df_drivers = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/drivers.csv",
    header=True,
    inferSchema=True
)

df_races = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/races.csv",
    header=True,
    inferSchema=True
)

df_results = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/results.csv",
    header=True,
    inferSchema=True
)

# select useful columns
drivers_q4 = df_drivers.select("driverId", "forename", "surname", "dob")
races_q4 = df_races.select("raceId", "date")
results_q4 = df_results.select("raceId", "driverId")

# join datasets
q4_df = results_q4.join(drivers_q4, on="driverId", how="inner") \
    .join(races_q4, on="raceId", how="inner")

# create Age column: age in completed years on the race date
q4_df = q4_df.withColumn(
    "Age",
    floor(datediff(col("date"), col("dob")) / 365.25)
)

# find youngest driver in each race
youngest_window = Window.partitionBy("raceId").orderBy(col("Age").asc())

youngest_df = q4_df.withColumn(
    "rank_youngest",
    row_number().over(youngest_window)
).filter(col("rank_youngest") == 1) \
 .select(
     "raceId",
     "date",
     "driverId",
     "forename",
     "surname",
     "Age"
 ).withColumnRenamed("driverId", "youngest_driverId") \
  .withColumnRenamed("forename", "youngest_forename") \
  .withColumnRenamed("surname", "youngest_surname") \
  .withColumnRenamed("Age", "youngest_age")

# find oldest driver in each race
oldest_window = Window.partitionBy("raceId").orderBy(col("Age").desc())

oldest_df = q4_df.withColumn(
    "rank_oldest",
    row_number().over(oldest_window)
).filter(col("rank_oldest") == 1) \
 .select(
     "raceId",
     "driverId",
     "forename",
     "surname",
     "Age"
 ).withColumnRenamed("driverId", "oldest_driverId") \
  .withColumnRenamed("forename", "oldest_forename") \
  .withColumnRenamed("surname", "oldest_surname") \
  .withColumnRenamed("Age", "oldest_age")

# combine youngest and oldest results
q4_final = youngest_df.join(oldest_df, on="raceId", how="inner") \
    .orderBy("raceId")

display(q4_final)

I first joined the results, drivers, and races datasets using raceId and driverId so that each driver could be associated with a specific race and its date. 

I defined Age as the driver’s age in completed years on the race date. To compute this, I used datediff() to calculate the number of days between the race date and the driver’s date of birth, then divided by 365.25 to convert days into years. I applied floor() to ensure that the age represents completed years rather than fractional values.

To identify the youngest and oldest drivers in each race, I used window functions partitioned by raceId. I ordered the data by Age in ascending order to find the youngest driver and in descending order to find the oldest driver. Then, I used row_number() to assign a rank within each race and selected the first row (rank = 1) to obtain the youngest and oldest drivers.

5.At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

To answer this question, I use the results dataset and track each driver’s podium finishes over time. I define podiums as finishing in position 1, 2, or 3. For each driver, I calculate the cumulative number of wins (position = 1), second places (position = 2), and third places (position = 3) up to each race. I use window functions partitioned by driverId and ordered by raceId to compute running totals for each podium type.

In [0]:
from pyspark.sql.functions import col, when, sum
from pyspark.sql.window import Window

df_results = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/results.csv",
    header=True,
    inferSchema=True
)

df_races = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/races.csv",
    header=True,
    inferSchema=True
)

df_results_q5 = df_results.select("raceId", "driverId", "position") \
    .withColumn(
        "position",
        when(col("position") == "\\N", None).otherwise(col("position"))
    ) \
    .withColumn("position", col("position").cast("int")) \
    .filter(col("position").isNotNull())

df_races_q5 = df_races.select("raceId", "date")

q5_df = df_results_q5.join(df_races_q5, on="raceId", how="inner")

window_spec = Window.partitionBy("driverId").orderBy("date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

q5_df = q5_df.withColumn(
    "wins",
    sum(when(col("position") == 1, 1).otherwise(0)).over(window_spec)
).withColumn(
    "second_places",
    sum(when(col("position") == 2, 1).otherwise(0)).over(window_spec)
).withColumn(
    "third_places",
    sum(when(col("position") == 3, 1).otherwise(0)).over(window_spec)
)

display(
    q5_df.select(
        "raceId",
        "date",
        "driverId",
        "position",
        "wins",
        "second_places",
        "third_places"
    ).orderBy("driverId", "date")
)

6.Continue exploring the data by answering your own question.

I choose Q6 question is : Which driver has the fastest average pit stop time overall?

To answer this, I use the pit_stops dataset and calculate each driver’s average pit stop time in milliseconds across all races. Then, I join the result with the drivers dataset to attach driver names. Finally, I sort the table from the smallest average pit stop time to the largest, so I can identify the fastest drivers overall in terms of pit stop time.

In [0]:
from pyspark.sql.functions import avg, count, concat_ws

# read datasets
df_pit = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/pit_stops.csv",
    header=True,
    inferSchema=True
)

df_drivers = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/drivers.csv",
    header=True,
    inferSchema=True
)
q6_df = df_pit.groupBy("driverId") \
    .agg(
        avg("milliseconds").alias("avg_pit_overall"),
        count("*").alias("num_pit_stops")
    )
q6_df = q6_df.join(
    df_drivers.select("driverId", "forename", "surname"),
    on="driverId",
    how="left"
)

q6_df = q6_df.withColumn(
    "driver_name",
    concat_ws(" ", "forename", "surname")
).select(
    "driverId",
    "driver_name",
    "avg_pit_overall",
    "num_pit_stops"
).orderBy("avg_pit_overall")

display(q6_df)

End
